In [2]:
import pandas as pd
import os, sys, django, dotenv
BASE_DIR = os.path.abspath(os.path.join(os.getcwd(), "..", ".."))
sys.path.append(BASE_DIR)
os.environ.setdefault("DJANGO_SETTINGS_MODULE", "tcc.settings")
django.setup()
dotenv.load_dotenv()
from myapi.models import Pessoa

In [ ]:
csv_path = os.path.join(BASE_DIR, 'pipelines', 'data', '202507_PEP.csv')
if os.path.exists(csv_path):
	df = pd.read_csv(csv_path, encoding='latin1', dtype=str, on_bad_lines='warn', sep=';')
else:
	print(f"Arquivo não encontrado: {csv_path}")

In [ ]:
df['Data_Início_Exercício'] = pd.to_datetime(df['Data_Início_Exercício'], format='%d/%m/%Y', errors='coerce')
df['Data_Fim_Exercício'] = pd.to_datetime(df['Data_Fim_Exercício'], format='%d/%m/%Y', errors='coerce')
df.rename(columns={
    'Nome_PEP': 'nome',
    'Descrição_Função': 'funcao',
    'Nome_Órgão': 'orgao',
    'Data_Início_Exercício': 'data_inicio_funcao',
    'Data_Fim_Exercício': 'data_fim_funcao',
    'CPF': 'cpf'
}, inplace=True)
df = df[['cpf', 'nome', 'funcao', 'orgao', 'data_inicio_funcao', 'data_fim_funcao']]
df

In [ ]:
output_path = os.path.join(BASE_DIR, 'pipelines', 'data', '202507_PEP.csv')
if not os.path.exists(output_path):
    df.to_csv(output_path, index=False, sep=';', encoding='latin1')

### Unindo os csvs consulta_cand_XX e bens_cand_XX

In [3]:
df_bens_SP = pd.read_csv(os.path.join(BASE_DIR, 'pipelines', 'data', 'bens_cand_SP.csv'), encoding='latin1', dtype=str, on_bad_lines='warn', sep=';')
df_bens_RJ = pd.read_csv(os.path.join(BASE_DIR, 'pipelines', 'data', 'bens_cand_RJ.csv'), encoding='latin1', dtype=str, on_bad_lines='warn', sep=';')
df_bens = pd.concat([df_bens_SP, df_bens_RJ], ignore_index=True)
df_consulta_SP = pd.read_csv(os.path.join(BASE_DIR, 'pipelines', 'data', 'consulta_cand_SP.csv'), encoding='latin1', dtype=str, on_bad_lines='warn', sep=';')
df_consulta_RJ = pd.read_csv(os.path.join(BASE_DIR, 'pipelines', 'data', 'consulta_cand_RJ.csv'), encoding='latin1', dtype=str, on_bad_lines='warn', sep=';')
df_consulta = pd.concat([df_consulta_SP, df_consulta_RJ], ignore_index=True)

In [4]:
df = pd.merge(df_bens[df_bens['CD_TIPO_BEM_CANDIDATO'] == '32'], df_consulta, on='SQ_CANDIDATO', how='inner')
lista_nomes = df['NM_CANDIDATO'].unique().tolist()

In [5]:
res = [nome for nome in lista_nomes if nome[0] == 'A']
res.sort()
res

['ABEL DE SOUZA REMONDI NETO',
 'ABEL HENRIQUE DUARTE',
 'ABEL MOURA DOS SANTOS',
 'ABIDIAS AUGUSTO DA SILVA',
 'ABNADAR REIS FILHO',
 'ABRAAO JOAO RODRIGUES',
 'ACACIO MACARIO DOS SANTOS',
 'ADAILTON CESAR MENOSSI',
 'ADAIR MOREIRA SANTOS RODRIGUES',
 'ADALBERTO ANTONIO ALVES DE SOUZA',
 'ADALBERTO DE OLIVEIRA BENTO',
 'ADALBERTO VITAL COELHO',
 'ADALGISA LOPES WARD',
 'ADALTO DE JESUS SILVA',
 'ADAURI DONIZETE DA SILVA',
 'ADAUTO ALEXANDRE CATELANI',
 'ADAUTO DE OLIVEIRA',
 'ADAUTO GONÇALVES PEREIRA',
 'ADAUTO MUNIZ DE ANDRADE',
 'ADELINO ALVES NETO',
 'ADELINO PESTANA GARCES',
 'ADEMAR CALEGÃO',
 'ADEMAR GUEDES SANTANA',
 'ADEMILSON APARECIDO SERVIDONE',
 'ADEMILSON DONIZETE MILITÃO',
 'ADEMILSON DONIZETI DOMINGOS',
 'ADEMILSON RAMOS DA SILVA',
 'ADEMOZAR DE CARVALHO',
 'ADENILSON GONÇALVES',
 'ADENOR CUNHA DA SILVA',
 'ADERMO DOS SANTOS NEVES',
 'ADHEMAR FERREIRA DE CAMARGO NETO',
 'ADIEL PEREIRA',
 'ADILSON ARMANDO CARVALHO AMADEU',
 'ADILSON BENEDITO PEREIRA',
 'ADILSON BEZERRA D

In [6]:
lista_pessoas = []
for pessoa in Pessoa.nodes:
    if pessoa.nome in res:
        lista_pessoas.append(pessoa.nome)
len(lista_pessoas)

5

In [7]:
lista_pessoas

['ADILSON PATROCINIO DOS SANTOS',
 'ADRIANO JOSE DA SILVA',
 'ALEX SPINELLI MANENTE',
 'ANA PAULA DA SILVA',
 'ANTONIO CARLOS RODRIGUES DE OLIVEIRA']

In [8]:
df = df[df['NM_CANDIDATO'].isin(lista_pessoas)]
df = df[['CD_TIPO_BEM_CANDIDATO', 'DS_TIPO_BEM_CANDIDATO', 'DS_BEM_CANDIDATO', 'SQ_CANDIDATO', 'NM_CANDIDATO', 'VR_BEM_CANDIDATO']]
df

,CD_TIPO_BEM_CANDIDATO,DS_TIPO_BEM_CANDIDATO,DS_BEM_CANDIDATO,SQ_CANDIDATO,NM_CANDIDATO,VR_BEM_CANDIDATO
1267,32,Quotas ou quinhões de capital,CAPITAL SOCIAL COOPERATIVA AGRICOLA MISTA DEAD...,250002024976,ADILSON PATROCINIO DOS SANTOS,"22592,82"
1268,32,Quotas ou quinhões de capital,CAPITAL SOCIAL COOPERATIVA DOS PLANTADORES DE ...,250002024976,ADILSON PATROCINIO DOS SANTOS,"3207,6"
1765,32,Quotas ou quinhões de capital,BANCO SICOOB CREDIÇÚCAR,250002024047,ANTONIO CARLOS RODRIGUES DE OLIVEIRA,"1240,38"
2283,32,Quotas ou quinhões de capital,EMPRESA,250002165877,ANA PAULA DA SILVA,30000
2284,32,Quotas ou quinhões de capital,EMPRESA,250002165877,ANA PAULA DA SILVA,5000
2448,32,Quotas ou quinhões de capital,PROPRIETARIA EMPRESA GAMES APS,250002045407,ANA PAULA DA SILVA,70000
3020,32,Quotas ou quinhões de capital,10% COTAS DA CONSTRUTORA ANEX,250002045641,ALEX SPINELLI MANENTE,31000
3021,32,Quotas ou quinhões de capital,10% COTAS DA CONSTRUTORA ANEX,250002045641,ALEX SPINELLI MANENTE,31000
3022,32,Quotas ou quinhões de capital,25% DE PARTICIPAÇÃO DE CAPITAL SOCIAL DA EMPRE...,250002045641,ALEX SPINELLI MANENTE,167500
3023,32,Quotas ou quinhões de capital,25% DE PARTICIPAÇÃO DE CAPITAL SOCIAL DA EMPRE...,250002045641,ALEX SPINELLI MANENTE,167500


In [43]:
path = os.path.join(BASE_DIR, 'pipelines', 'data', 'bens_consultas_RJ_SP.csv')
if os.path.exists(path):
    os.remove(path)
df.to_csv(path, index=False, sep=';', encoding='latin1')